# DATA-445: In-Class Assignment 10

Consider the `framingham.csv` data file. The dataset is publicly available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are demographic, behavioral, and medical risk factors.

**Demographic:**
- Sex: male or female (Nominal)
- Age: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

**Behavioral:**
- Current Smoker: whether or not the patient is a current smoker (Nominal)
- Cigs Per Day: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette)

**Medical (history):**
- BP Meds: whether or not the patient was on blood pressure medication (Nominal)
- Prevalent Stroke: whether or not the patient had previously had a stroke (Nominal)
- Prevalent Hyp: whether or not the patient was hypertensive (Nominal)
- Diabetes: whether or not the patient had diabetes (Nominal)

**Medical (current):**
- Tot Chol: total cholesterol level (Continuous)
- Sys BP: systolic blood pressure (Continuous)
- Dia BP: diastolic blood pressure (Continuous)
- BMI: Body Mass Index (Continuous)
- Heart Rate: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values)
- Glucose: glucose level (Continuous)

**Predict variable (desired target):**
- 10 year risk of coronary heart disease CHD (binary: "1", means "Yes", "0" means "No")

In **Python**, answer the following:

## 1. (3 points)
Using the pandas library, read the csv data file and create a data-frame called `heart`.

In [1]:
import pandas as pd 

heart = pd.read_csv("framingham.csv")
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## 2. (3 points)
Remove observations with missing values.

In [2]:
heart = heart.dropna().reset_index(drop=True)
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## 3. (4 points)
Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` is the target variable. Define a 5-fold cross validation strategy (taking into account the proportion of 0s and 1s).

In [3]:
from sklearn.model_selection import StratifiedKFold

X = heart[["age", "currentSmoker", "totChol", "BMI", "heartRate"]]
y = heart["TenYearCHD"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 4. (8 points)
Train and evaluate the logistic regression model over the five folds. Using 10% as cutoff value, report the recall. Make sure to 0-1 scale the input features and build the model with `MinMaxScaler` and `make_pipeline`.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import recall_score

md = make_pipeline(MinMaxScaler(),
                   LogisticRegression(random_state=42))

recall_scores = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    md.fit(X_train, y_train)
    md_pred = md.predict_proba(X_test)[:, 1]
    md_pred_label = (md_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, md_pred_label)
    recall_scores.append(score)
    print(f"Fold {i+1} Recall Score: {score}")
    
print(f"Average Recall Score: {sum(recall_scores)/len(recall_scores)}")
    

Fold 1 Recall Score: 0.75
Fold 2 Recall Score: 0.8288288288288288
Fold 3 Recall Score: 0.9369369369369369
Fold 4 Recall Score: 0.9099099099099099
Fold 5 Recall Score: 0.8482142857142857
Average Recall Score: 0.8547779922779922


## 5. (8 points)
Using the model from part 4, and `BaggingClassifier` build a model with 20 estimators. Using 10% as cutoff value, report the recall.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import recall_score

bagging_md = make_pipeline(MinMaxScaler(),
                           BaggingClassifier(estimator=LogisticRegression(), 
                                             n_estimators=20, 
                                             random_state=42))

recall_scores = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    bagging_md.fit(X_train, y_train)
    bagging_pred = bagging_md.predict_proba(X_test)[:, 1]
    bagging_pred_label = (bagging_pred >= 0.1).astype(int)
    
    score = recall_score(y_test, bagging_pred_label)
    recall_scores.append(score)
    print(f"Fold {i+1} Recall Score: {score}")
    
print(f"Average Recall Score: {sum(recall_scores)/len(recall_scores)}")

Fold 1 Recall Score: 0.7678571428571429
Fold 2 Recall Score: 0.8198198198198198
Fold 3 Recall Score: 0.9279279279279279
Fold 4 Recall Score: 0.9279279279279279
Fold 5 Recall Score: 0.8482142857142857
Average Recall Score: 0.8583494208494209


## 6. (3 points)
Using the results from parts (4) and (5), what model would you use to predict `TenYearCHD` based on the recall?

Based on the results, I would the the model from part (5) to predict `TenYearCHD` because its recall is greater.